In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r"D:\AI Eng\Projects\RNN-Sentiment Analysis\data\IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

### Pre-processing

#### Removing URLs

In [8]:
import re # Regular Expressions
def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text
df["review"]=df["review"].apply(remove_urls)

#### Removing Punctuations

In [9]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text
df["review"]=df["review"].apply(remove_punctuation)

#### Removing HTML

In [10]:
def removing_html(text):
    text=re.sub(r"<.>","",text)
    return text
df["review"]=df["review"].apply(removing_html)

In [11]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production br br The filmin...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically theres a family where a little boy J...,negative
4,Petter Matteis Love in the Time of Money is a ...,positive


#### Removing the Stopswords

In [12]:
!pip install nltk
import nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")
# this all are modules

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dexter\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dexter\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dexter\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
df["review"]=df["review"].apply(remove_stopwords)

#### Stemming

In [16]:
from nltk.stem import PorterStemmer

In [17]:
def stemming (text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)


#### Encoding the target value

In [18]:
from sklearn.preprocessing import LabelEncoder

In [19]:
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [20]:
Y=df["sentiment"]
Y.head()

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

#### Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

In [23]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4306699 stored elements and shape (49582, 5000)>

In [24]:
df.head()

,review,sentiment
0,one review nti wtchg 1 oz epod ll hook they ri...,1
1,a wder ltle producti br br t film techniqu uns...,1
2,i thought wder wy spend time o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


### Dataset and DataLoader

In [34]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [35]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [37]:
x_train=x_train.toarray()
x_test=x_test.toarray()

In [42]:
train_set=TensorDataset
(
    torch.tensor(x_train).float(),
    torch.tensor(y_train.values).float()   # this create a copy of array
)

(tensor([[0.2036, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]),
 tensor([0., 0., 1.,  ..., 0., 1., 1.]))

In [ ]:
# train_setTensorDataset
# (
#     torch.from_numpy(x_train).float(),
#     torch.from_numpy(y_train.values).float()   # this create a connections
# )

In [ ]:
test_set=TensorDataset
(
    torch.tensor(x_test).float(),
    torch.tensor(y_test.values).float()   # this create a copy of array
)